# 14b - Lender identity from the Charges API

This notebook closes the single biggest gap in the whole pipeline. Everything in
`panel_deltas` describes the **company**: how many charges it holds, whether it files on
time, whether it is growing. Nothing in it describes **who banks the company**. But the
three macro areas Lloyds actually asked about are defined on exactly that axis:

| Area | What it means | What we could say before this notebook |
|---|---|---|
| Growth | not our client, will need a product | we can predict need, but not tell ours from theirs |
| Attrition | our client, and leaving | nothing at all |
| Maintenance | our client, and staying | nothing at all |

The unlock is that Companies House charges are **append-only and dated**. Every charge
carries `created_on`, `satisfied_on`, a status, and `persons_entitled[].name`, which is the
lender. So one API pass today reconstructs the entire 33-month lender history
retrospectively, with no historical downloads at all. That is what I ran, and it took a day.

**This is a feature-building notebook**, so it belongs with 13 and 14 rather than with 15. It
produces explanatory variables that get installed into the modelling matrix. It does also
build a fifth label (`switching`), because the same data happens to answer a question the bulk
file cannot, but that label is measured and then deliberately left untrained; section 7 is the
decision and the numbers behind it.

What is in here, in order:

1. **Proof of the harvest** (step 0 of the plan): the log, the shards, the counts.
2. **The lender taxonomy** (step 2): collapsing 79k free-text names onto institutions.
3. **The point-in-time lender panel** (step 3): who held what, month by month, and what each
   of the thirteen new columns means.
4. **Verification**: the seven checks the plan asks for, run here, pass or fail.
5. **Integration** (step 4): the lender features joined onto the modelling matrices, plus
   the fifth `switching` label.
6. **What it delivers to the client**: three rule-based feeds, no model required.
7. **The decision on `switching`**: features yes, target no, with the trade-off costed.
8. **Where this leaves us**, including why none of notebook 16's saved results moved.

One caveat that belongs on every slide this ever reaches: **charges see secured lending
only**. Overdrafts, cards, current accounts, merchant services and unsecured loans are
invisible to Companies House. So what I am calling attrition here is attrition of the
*lending* relationship, which is a proxy for the banking relationship and not the same thing.

## Setup

Everything below runs from the repo root, same convention as the other notebooks.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import duckdb
import pandas as pd

from src.features import charges, lenders, panel
from src.models import targets

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("PRAGMA threads=6")

LENDER_GLOB = (charges.LENDER_PANEL_DIR / "**" / "*.parquet").as_posix()
DELTA_GLOB = (panel.DELTA_DIR / "**" / "*.parquet").as_posix()
print(f"panel months: {panel.FIRST_MONTH} to {panel.LAST_MONTH}")

panel months: 2023-10-01 to 2026-07-01


## 1. The harvest, and proof that it ran

I launched it fully detached on the evening of 26 July, deliberately **not** as a
background job owned by the tooling:

```bash
setsid nohup .venv/bin/python -m src.features.charges harvest \
    > logs/charges_harvest.log 2>&1 < /dev/null &
```

Detached means closing the terminal cannot kill it, and all progress goes to a log file
rather than into anybody's context window, so a 23-hour job costs nothing while it runs.
It is resumable from the shards themselves rather than from a side ledger, which is the
property that matters when a job this long gets interrupted: the data *is* the bookkeeping,
so the two cannot drift apart.

The target list was every company that has held at least one charge in **any** of the 33
panel snapshots (`Mortgages.NumMortCharges > 0`), which is 158,684 companies out of the
2,038,130 in the universe, or 7.8%. I deliberately did not restrict it to companies with a
charge outstanding *today*: a company that has paid off its last charge and taken nothing new
is precisely an exit candidate, so filtering those out would have deleted the signal I was
harvesting for.

Here is the log, first and last lines.

In [2]:
log_path = Path("logs/charges_harvest.log")
lines = log_path.read_text().splitlines()

print(f"{log_path} ({len(lines)} lines)\n")
for line in lines[:3]:
    print(line)
print("   ...")
for line in lines[-3:]:
    print(line)

started = pd.Timestamp(lines[0][1:20])
finished = pd.Timestamp(lines[-1][1:20])
print(f"\nran {started} -> {finished}  ({(finished - started).total_seconds() / 3600:.1f} hours)")
print(f"errors/retries logged: {sum('retry' in ln or '429' in ln or 'SKIP' in ln for ln in lines)}")

logs/charges_harvest.log (162 lines)

[2026-07-26 18:35:39] targets=158,684 already_done=0 todo=158,684 eta=22.9h
[2026-07-26 18:36:51] targets=158,684 already_done=61 todo=158,623 eta=23.4h
[2026-07-26 18:46:02] 1,000/158,623 (1.81/s, 24.1h left)
   ...
[2026-07-27 19:47:07] 157,000/158,623 (1.73/s, 0.3h left)
[2026-07-27 19:56:00] 158,000/158,623 (1.73/s, 0.1h left)
[2026-07-27 20:01:38] done, 158,684 companies on disk

ran 2026-07-26 18:35:39 -> 2026-07-27 20:01:38  (25.4 hours)
errors/retries logged: 1


25.4 hours wall clock at a sustained 1.7 to 1.8 requests per second, which is just under the
Companies House ceiling of 2/s. The pacing is on the interval between request *starts* rather than a
fixed `sleep` after each one, because a fixed sleep quietly adds the network latency on top:
measured, a 0.52s sleep gave 1.5 req/s and a 29-hour run instead of 23.

Now the data itself. One NDJSON shard per 5,000 companies, one JSON record per company,
charges nested inside it.

In [3]:
shards = sorted(Path("data/raw/charges").glob("part-*.jsonl"))
total_mb = sum(s.stat().st_size for s in shards) / (1 << 20)
print(f"{len(shards)} shards, {total_mb:,.0f} MB total")
print(f"first: {shards[0]}   last: {shards[-1]}")

RAW_GLOB = "data/raw/charges/part-*.jsonl"
harvest_counts = con.execute(f"""
    SELECT count(*)                       AS records,
           count(DISTINCT company_number)  AS companies,
           sum(not_found::INT)             AS not_found_404,
           sum(len(charges))               AS charges,
           min(fetched_at)                 AS first_fetch,
           max(fetched_at)                 AS last_fetch
    FROM read_json('{RAW_GLOB}', format='newline_delimited', maximum_object_size=50000000)
""").df()
harvest_counts.T

32 shards, 367 MB total
first: data/raw/charges/part-0000.jsonl   last: data/raw/charges/part-0031.jsonl


,0
records,158684
companies,158684
not_found_404,0.0
charges,558693.0
first_fetch,2026-07-26 17:35:39.842520
last_fetch,2026-07-27 19:01:38.829079


**158,684 companies, 558,693 charges, zero 404s and zero duplicates.** That is verification 1
from the plan, and it passes exactly: every targeted company came back, and because resume is
derived from the shards, re-running the harvest now would fetch nothing.

One raw record, so the team can see what we are actually working from. (`charge_number` is
per-company sequential and present on every charge, unlike `charge_code`, which only exists
for the roughly 44% filed since the 2013 regime, so the charge key is built from the former.
Four company-charge numbers collide across the whole harvest, which is why the distinct-charge
count below is 558,689 rather than 558,693.) Note `created_on`,
`satisfied_on` and `persons_entitled`, which are the three fields the entire rest of this
notebook is built on.

In [4]:
with open(shards[0]) as fh:
    example = json.loads(fh.readline())

print(f"company {example['company_number']}, fetched {example['fetched_at']}, "
      f"{len(example['charges'])} charges\n")
print(json.dumps(example["charges"][0], indent=2)[:900])

company 00000121, fetched 2026-07-26T17:35:39.842520+00:00, 4 charges

{
  "charge_number": 4,
  "status": "outstanding",
  "created_on": "2012-03-09",
  "delivered_on": "2012-03-14",
  "classification": {
    "type": "charge-description",
    "description": "Charge over investment portfolio and credit balances"
  },
  "particulars": {
    "type": "short-particulars",
    "description": "All rights, benefits and interests to and under the investment plans, all deposits, all securities, all chattels, preciuos metals and all other assets see image for full details."
  },
  "persons_entitled": [
    {
      "name": "Barclays Bank PLC, Singapore Branch"
    }
  ],
  "secured_details": {
    "type": "amount-secured",
    "description": "All monies due or to become due from the company to the chargee on any account whatsoever under the terms of the aforementioned instrument creating or evidencing the charge"
  }
}


## 2. The lender taxonomy

558,693 charges carry 572,776 `persons_entitled` rows across 78,708 distinct names (a charge
can have more than one lender on it, and 443 charges name nobody at all). Those names are free text typed by whoever filed the MR01, so "Lloyds Bank
plc", "Lloyds Bank PLC as security agent" and "Lloyds Bank Commercial Finance Limited" are
three strings and one bank. Until they are collapsed onto institutions, nothing can ask "is
this our client".

I built this as an **ordered dictionary of regex rules**, in `src/features/lenders.py`, not as
a fuzzy-matching problem. Bank lending is extremely concentrated: the top four groups alone
are more than half of all charges. `matching.py` exists for the contract-supplier problem
where the join key is a company name with no register behind it, and it earns its complexity
there. Here the population of lenders is small, stable and knowable, so an explicit rule list
is cheaper and, more importantly, auditable. A client can read the rule that decided their
company is ours.

Three ordering decisions do the real work, since the rules are first-match-wins:

1. `lbg_retail` runs **before** `lbg`. Halifax, Black Horse, MBNA, Scottish Widows and Lex
   Autolease are Lloyds Banking Group but they are not commercial lending, so the plan draws
   the boundary at commercial entities. They get their own visible group rather than being
   silently counted as ours or silently dropped.
2. `natwest` runs **before** `lbg`, so "The Royal Bank of Scotland plc" cannot be read as
   Bank of Scotland. The `lbg` pattern also carries a negative lookbehind for exactly that
   case, so the two defences are independent and a future reordering cannot quietly hand
   RBS's book to Lloyds.
3. Bank groups run **before** the trustee/SPV catch-all, so "Barclays Security Trustee
   Limited" is Barclays and only genuine third parties (GLAS, Wilmington, Kroll) land in
   `trustee_spv`.

Legacy names map to whoever owns the book today, because charges here go back to the 1990s
and the question is "who does this company borrow from", not "what was that entity called at
the time": Midland Bank to HSBC, Lloyds TSB to LBG, Abbey National to Santander, Yorkshire
Bank to Virgin Money.

In [5]:
%%time
# Explodes the shards to (charge, lender) grain and classifies every name.
flat = charges.build_flat()
print(f"{len(flat):,} (charge, lender) rows, {flat['charge_id'].nunique():,} distinct charges, "
      f"{flat['lender_name'].nunique():,} distinct lender names")
flat.head(3)

572,776 (charge, lender) rows, 558,689 distinct charges, 78,708 distinct lender names
CPU times: user 16.6 s, sys: 1.59 s, total: 18.2 s
Wall time: 11.1 s


,CompanyNumber,charge_id,charge_number,charge_code,status,created_on,delivered_on,satisfied_on,classification,lender_name,fetched_at,lender_group,is_lbg
0,00000121,00000121-4,4,NaN,outstanding,2012-03-09,2012-03-14,NaT,Charge over investment portfolio and credit ba...,"Barclays Bank PLC, Singapore Branch",2026-07-26 17:35:39.842520,barclays,False
1,00000121,00000121-3,3,NaN,outstanding,2005-07-27,2005-08-05,NaT,Memorandum of charge,Ing Asia Private Bank Limited,2026-07-26 17:35:39.842520,other_bank,False
2,00000121,00000121-2,2,NaN,fully-satisfied,1997-07-01,1997-07-12,1997-08-08,Cash collateral agreement,Standard Bank London Limited,2026-07-26 17:35:39.842520,other_bank,False


In [6]:
coverage = lenders.coverage(flat)
coverage.style.format({"rows": "{:,}", "share": "{:.2%}"})

,rows,share
lender_group,,
unclassified,"128,666",22.46%
natwest,"93,927",16.40%
lbg,"69,408",12.12%
barclays,"68,968",12.04%
hsbc,"68,270",11.92%
other_bank,"34,250",5.98%
challenger_bank,"30,707",5.36%
trustee_spv,"26,496",4.63%
asset_invoice_finance,"23,127",4.04%


**77.5% of charge-lender rows fall into a named group**, so unclassified is 22.5%, comfortably
inside the 30% the plan set as the line where the dictionary is not done. NatWest is far and
away the largest holder in this population at 16.4%, then LBG at 12.1%, Barclays at 12.0% and
HSBC at 11.9%. Lloyds is second on this measure and only a hair ahead of the other two, so
**the big four hold about 52% of charges in the BB/SME register and NatWest holds a third more
of them than we do**. That is a finding for the growth conversation on its own.

`lbg_retail` is 340 rows out of 573k, so the retail exclusion changes almost nothing
numerically. I kept it anyway because the client will ask where the boundary is, and
"we drew it explicitly and here is what it cost" is a better answer than a number.

The unclassified tail is not a failure, it is the long tail of individuals, landlords,
directors lending to their own company, and one-off SPVs. Those are real charges with no
institution behind them. Here are the biggest ones still uncaught.

In [7]:
lenders.top_unclassified(flat, 15)

,lender_name,charges
0,Alfandari Private Equities Limited,247
1,Alfandari Private Equities LTD,204
2,British Broadcasting Corporation,159
3,Ingenious Resources Limited,148
4,Head Gear Films Fn LTD.,143
5,New Millennium Investors L.L.C.,133
6,"Film Finances, Inc.",133
7,The British Film Institute,133
8,"The Connaught Income Fund, Series 1",133
9,Davenham Trade Finance Limited,132


Nothing left in the top of that list is a bank. `Alfandari Private Equities` is a family
lending vehicle, `Ingenious Resources` and `Head Gear Films` are film-financing SPVs, the BBC
is the BBC. These are genuinely unclassifiable rather than missing, which is the signal that
the dictionary has reached the end of its useful returns.

A few spot checks on the cases the ordering exists to protect.

In [8]:
pd.DataFrame([
    {"name": n, "group": lenders.classify(n)} for n in [
        "Lloyds Bank plc",
        "Lloyds Bank Commercial Finance Limited",
        "LDC (Managers) Limited",
        "The Governor and Company of the Bank of Scotland",
        "Bank of Wales PLC",
        "The Royal Bank of Scotland plc",
        "Halifax plc",
        "Barclays Security Trustee Limited",
        "GLAS Trust Corporation Limited",
        "Mr John Smith",
    ]
])

,name,group
0,Lloyds Bank plc,lbg
1,Lloyds Bank Commercial Finance Limited,lbg
2,LDC (Managers) Limited,lbg
3,The Governor and Company of the Bank of Scotland,lbg
4,Bank of Wales PLC,lbg
5,The Royal Bank of Scotland plc,natwest
6,Halifax plc,lbg_retail
7,Barclays Security Trustee Limited,barclays
8,GLAS Trust Corporation Limited,trustee_spv
9,Mr John Smith,unclassified


## 3. The point-in-time lender panel

Now the replay. A charge is **outstanding at month `t`** if and only if it was created on or
before `t` and either has no satisfaction date or was satisfied after `t`. That one predicate,
run against every month of the panel, reconstructs the lender relationship history without a
single extra API call.

Two decisions worth flagging to the team:

**The as-of gate is the first of the month, not the last.** The contract table gates on
`last_day(snapshot_date)` because contracts are dated by publication and that is the natural
month-end question. Charges are different: the panel row for month `t` is the register as it
stood at the *start* of `t`, so anything a charge does later in the month is genuinely not
knowable at `t`. Gating on month end would put up to 30 days of future charge activity into a
feature row, which is exactly what verification 7 exists to catch.

**June 2025 needs no special case here.** This table is built from charge event dates over a
full calendar spine, not from register snapshots, so the missing snapshot is simply a month
the panel never asks about. That is the same reason the contract as-of table needs no
special case, and it is the opposite of the delta features, where a positional lag would have
silently spanned the hole.

The output lands in `data/processed/lender_panel/snapshot_date=YYYY-MM-01/`, which is the
identical partition layout to `contracts_asof/`, so it joins the same way with the same
coalesce convention.

In [9]:
%%time
n_rows = charges.build_lender_panel(threads=6)
print(f"{n_rows:,} company-month rows across {len(list(charges.LENDER_PANEL_DIR.iterdir()))} months")
print(f"features: {charges.LENDER_FEATURE_COLS + charges.LENDER_CATEGORICAL_COLS}")

5,147,067 company-month rows across 34 months
features: ['n_charges_outstanding', 'n_lbg_charges_outstanding', 'is_lbg_client', 'lbg_share_of_outstanding', 'n_distinct_lenders', 'n_competitor_lenders', 'months_since_last_lbg_charge_created', 'months_since_last_lbg_satisfaction', 'competitor_entered_12m', 'lbg_charge_satisfied_6m', 'competitor_charge_created_6m', 'ever_lbg_client', 'primary_lender_group']
CPU times: user 1min 19s, sys: 25 s, total: 1min 44s
Wall time: 19.3 s


In [10]:
con.execute(f"""
    SELECT * FROM read_parquet('{LENDER_GLOB}')
    WHERE snapshot_date = DATE '2026-07-01' AND is_lbg_client AND n_distinct_lenders > 1
    ORDER BY n_charges_outstanding DESC LIMIT 5
""").df()

,CompanyNumber,n_charges_outstanding,n_lbg_charges_outstanding,is_lbg_client,lbg_share_of_outstanding,n_distinct_lenders,n_competitor_lenders,months_since_last_lbg_charge_created,months_since_last_lbg_satisfaction,competitor_entered_12m,lbg_charge_satisfied_6m,competitor_charge_created_6m,ever_lbg_client,primary_lender_group,snapshot_date
0,02634371,1230,225,True,0.182927,12,10,69,8,False,False,False,True,natwest,2026-07-01
1,02630203,877,145,True,0.165336,11,10,6,1,False,True,True,True,natwest,2026-07-01
2,02178783,738,116,True,0.157182,10,8,45,2,False,True,True,True,natwest,2026-07-01
3,01846413,682,107,True,0.156891,11,10,54,3,False,True,False,True,natwest,2026-07-01
4,03094287,614,92,True,0.149837,11,10,3,6,False,True,True,True,natwest,2026-07-01


### What each of those columns is

Every one of these is derived from three raw fields and nothing else: `created_on`,
`satisfied_on` and the `lender_group` that the taxonomy assigned to `persons_entitled[].name`.
"Outstanding at `t`" always means the same predicate, `created_on <= t AND (satisfied_on IS
NULL OR satisfied_on > t)`. "Competitor" always means `lender_group NOT IN ('lbg',
'lbg_retail', 'unclassified')`, so a Halifax charge or a director's personal loan is never
counted as a rival bank.

| Feature | Derived from | What it measures | How to read it |
|---|---|---|---|
| `n_charges_outstanding` | count of charges outstanding at `t` | total secured borrowing relationships live that month | size of the company's secured borrowing footprint. The denominator for everything else |
| `n_lbg_charges_outstanding` | same count, restricted to `is_lbg` | how much of that is ours | 0 means not a lending client this month, whatever the history |
| `is_lbg_client` | `n_lbg_charges_outstanding > 0` | are we in the book at `t` | the point-in-time client flag. This is the one that flips when a relationship ends |
| `lbg_share_of_outstanding` | LBG count / total count | wallet share of secured lending | 1.0 is sole banker, 0.2 is one charge in five. Falling share is the encroachment signal. NULL when the company has no charges at all, because there is no share to take |
| `n_distinct_lenders` | distinct `lender_group` outstanding, `unclassified` excluded | how many institutions are in the deal | 1 is a sole-banked company (defensible, and vulnerable if it is not us), 5+ is a syndicated or serially refinanced borrower |
| `n_competitor_lenders` | same, restricted to competitors | how many rivals are already inside | 0 with `is_lbg_client` true is the strongest relationship state we can observe |
| `months_since_last_lbg_charge_created` | `t` minus the newest LBG `created_on` | recency of the last time we lent | small is a live, recently transacting relationship. Large means the relationship is running off. NULL means we have never lent to them |
| `months_since_last_lbg_satisfaction` | `t` minus the newest LBG `satisfied_on` on or before `t` | recency of the last time one of our charges was released | 1 to 6 with `is_lbg_client` false is a just-departed client, which is exactly the switch feed. NULL means nothing of ours has ever been released |
| `competitor_entered_12m` | LBG-only at `t-12m` (`n_outstanding_12m_ago > 0`, none of them non-LBG) and a competitor charge created since | a rival breaking into an account that was ours alone | the early-warning flag. It fires while the company is still a client, which is the point: it gives an RM lead time rather than a post-mortem |
| `lbg_charge_satisfied_6m` | any LBG charge with `satisfied_on` in `(t-6m, t]` | we lost a facility recently | on its own it is ambiguous: repayment and defection look identical. Read it with the next one |
| `competitor_charge_created_6m` | any competitor charge created in `(t-6m, t]` | someone else lent to them recently | the two `_6m` flags together are the switch signature. Only one of them firing is deleveraging or additional borrowing, not a switch |
| `ever_lbg_client` | any LBG charge exists at all, satisfied or not | were we ever in this book | the win-back universe. `ever_lbg_client AND NOT is_lbg_client` is a lapsed client, which is a warmer call than a cold prospect |
| `primary_lender_group` | most outstanding charges at `t`, ties broken by the most recent `created_on` | who the company mainly banks with | the categorical the model one-hots. NULL when nothing is outstanding, or when everything outstanding is unclassified |

Two conventions to keep in mind when reading them in the matrix rather than here. The counts
and flags **coalesce to 0/false** for companies with no charge history at all, because "never
borrowed" genuinely is zero borrowing. The two `months_since_*` columns and
`lbg_share_of_outstanding` stay **NULL**, because there is no honest zero for "this has never
happened". That is the same split the contract features already use, so LightGBM's native
missing handling treats them the same way it already does there.

Reading the five rows above: these are the largest multi-lender LBG clients in the register.
Every one of them is majority-banked by NatWest with Lloyds holding roughly a sixth of the
charges, and four of the five have both `lbg_charge_satisfied_6m` and
`competitor_charge_created_6m` set. On a smaller company that pattern would be the switch
signature; on a 1,230-charge borrower it is ordinary refinancing churn, which is why the feed
further down uses the flags plus `NOT is_lbg_client` rather than the flags alone.

## 4. Verification

The plan lists seven checks. I am running all of them here rather than trusting the code,
because every one of them corresponds to a way this could be wrong while still looking
plausible.

### Check 2: do the charge counts reconcile with the bulk file?

This is the equivalent of the panel parity check from notebook 12 and it is the one that
matters most. The bulk file already carries `Mortgages.NumMortOutstanding` per company, and I
have just recomputed the same number from the API from scratch. If my `satisfied_on` handling
is wrong, these two disagree.

In [11]:
recon = con.execute(f"""
    SELECT count(*)                                                            AS companies,
           avg((l.n_charges_outstanding = p."Mortgages.NumMortOutstanding")::INT) AS exact_match,
           avg((abs(l.n_charges_outstanding - p."Mortgages.NumMortOutstanding") <= 1)::INT)
                                                                               AS within_one,
           avg((l.n_charges_outstanding > p."Mortgages.NumMortOutstanding")::INT) AS api_higher,
           avg((l.n_charges_outstanding < p."Mortgages.NumMortOutstanding")::INT) AS api_lower
    FROM read_parquet('{LENDER_GLOB}') l
    JOIN read_parquet('{(panel.PANEL_DIR / "snapshot_date=2026-07-01" / "*.parquet").as_posix()}') p
      USING ("CompanyNumber")
    WHERE l.snapshot_date = DATE '2026-07-01'
""").df()

assert recon["exact_match"].iloc[0] > 0.95, "charge counts do not reconcile with the bulk file"
print("PASS: the API-derived outstanding count matches the bulk file for "
      f"{recon['exact_match'].iloc[0]:.1%} of companies.")
recon.T

PASS: the API-derived outstanding count matches the bulk file for 97.7% of companies.


,0
companies,143269.000000
exact_match,0.977336
within_one,0.992783
api_higher,0.022035
api_lower,0.000628


**97.7% exact, 99.3% within one charge.** Worth spelling out what the 2.2% that disagrees
actually is, because "the two sources of truth disagree" sounds worse than it is.

The disagreement is almost entirely one-directional. Of the 143,269 companies compared:

- 2.20% have the **API count higher** than the bulk file,
- 0.06% have the API count **lower**.

That asymmetry is the whole diagnosis. Take a company with three charges: two plainly
outstanding, one filed as `part-satisfied` (some of the security released, the rest still
registered). My predicate asks "is there a `satisfied_on` date on or before `t`", and a
part-satisfied charge has none, so I count 3. The bulk file's `Mortgages.NumMortOutstanding`
counts 2. Same register, same month, different convention, and 415 charges in the harvest are
in that state. The second contributor is timing: the bulk file is a monthly extract, so a
charge created on 20 July appears in my count for 2026-07 (I read its `created_on`) and may
not yet appear in a bulk file cut earlier in the month. Both mechanisms can only ever push the
API number **up**.

The direction is what makes this safe rather than dangerous. An API count that is too *low*
would mean I am failing to see charges that exist, which for a feature called
`n_lbg_charges_outstanding` means quietly declaring live clients to be ex-clients, and it
would poison both the switching label and the switch feed. That failure mode is 0.06%, six
companies in ten thousand, and at that rate it is indistinguishable from ordinary filing
noise. The failure mode I do have is over-counting a released-but-not-fully-released charge by
one, which shifts a company's `lbg_share_of_outstanding` from, say, 2/4 = 0.50 to 3/5 = 0.60
and changes nothing about whether they are a client.

The one place it does bite is the reconciliation itself, so I use the API count everywhere
downstream rather than mixing the two: `Mortgages.NumMortOutstanding` from the bulk file stays
in the matrix as its own feature, and my `n_charges_outstanding` sits beside it. They are
99.3% the same number and the model is welcome to use whichever it prefers.

> **Added 7 August 2026, after the leak was found.** Read that "second contributor"
> sentence again: *"a charge created on 20 July appears in my count for 2026-07 (I read
> its `created_on`) and may not yet appear in a bulk file cut earlier in the month"*.
>
> That is not a reconciliation quirk. That is the leak, written out in full, five cells
> above the test that was supposed to catch it and did not. I had measured it, diagnosed
> the mechanism correctly, and then filed it under "safe rather than dangerous", because
> I was asking whether the **count** was wrong rather than whether the **clock** was. The
> count was fine. The clock was about three weeks fast, and every lender feature in the
> matrix was reading it.
>
> The direction argument is what did the damage. "Both mechanisms can only ever push the
> API number up" is true, and it is precisely why this was leakage rather than noise: the
> API number is too high exactly for the companies that were about to take a charge,
> which is the positive class of the `lending` label. A one-directional error correlated
> with the outcome is not a benign error, it is a feature.
>
> The evidence was on the screen in this cell and I read past it. For the write-up that
> is worth more than the fix is.


### Check 3: is it actually point-in-time?

Pick a company whose single Lloyds charge was satisfied in the middle of the panel window,
then walk the months across that date. `is_lbg_client` must flip from 1 to 0 in the month
*after* the satisfaction, and not before.

In [12]:
subject = con.execute(f"""
    SELECT "CompanyNumber", max(satisfied_on) AS satisfied_on
    FROM read_parquet('{charges.FLAT_PATH.as_posix()}')
    WHERE is_lbg
    GROUP BY 1
    HAVING count(*) = 1 AND max(satisfied_on) BETWEEN DATE '2024-06-01' AND DATE '2025-12-01'
    ORDER BY "CompanyNumber" LIMIT 1
""").df().iloc[0]

walk = con.execute(f"""
    SELECT snapshot_date, is_lbg_client, n_lbg_charges_outstanding, n_charges_outstanding,
           months_since_last_lbg_satisfaction, primary_lender_group
    FROM read_parquet('{LENDER_GLOB}')
    WHERE "CompanyNumber" = '{subject.CompanyNumber}'
      AND snapshot_date BETWEEN DATE '{subject.satisfied_on:%Y-%m-%d}' - INTERVAL 3 MONTH
                            AND DATE '{subject.satisfied_on:%Y-%m-%d}' + INTERVAL 3 MONTH
    ORDER BY 1
""").df()

flip = walk[walk["is_lbg_client"] == False]["snapshot_date"].min()
expected = (pd.Timestamp(subject.satisfied_on) + pd.offsets.MonthBegin(1))
assert flip == expected, f"flipped at {flip}, expected {expected}"
print(f"company {subject.CompanyNumber}, LBG charge satisfied {subject.satisfied_on:%Y-%m-%d}")
print(f"PASS: is_lbg_client flips at {flip:%Y-%m}, the first month after the satisfaction.")
walk

company 00138071, LBG charge satisfied 2024-09-30
PASS: is_lbg_client flips at 2024-10, the first month after the satisfaction.


,snapshot_date,is_lbg_client,n_lbg_charges_outstanding,n_charges_outstanding,months_since_last_lbg_satisfaction,primary_lender_group
0,2024-07-01,True,1,1,<NA>,lbg
1,2024-08-01,True,1,1,<NA>,lbg
2,2024-09-01,True,1,1,<NA>,lbg
3,2024-10-01,False,0,0,1,NaN
4,2024-11-01,False,0,0,2,NaN
5,2024-12-01,False,0,0,3,NaN


### Check 4: the June 2025 hole

The plan asks me to assert that 12-month lender deltas go NULL around the missing month. Having
built this, that expectation is wrong for **this** table and I want to say so rather than fake
a passing test.

The deltas in `panel_deltas` are lags over *register snapshots*, so a missing snapshot is a
missing input and NULL is the honest answer. The lender panel is computed from *charge event
dates* over a complete calendar spine, so June 2025 is a perfectly ordinary month here: the
charges that existed in June 2025 are known exactly, because I am reading their creation and
satisfaction dates, not a monthly extract. The right check is therefore the opposite one:
June 2025 exists, and it is smooth against its neighbours rather than a hole or a spike.

The panel never asks for that month anyway (it has no partition there), and the quarterly
origin grid steps from 2025-04 to 2025-07 straight over it.

**So how are we handling the hole overall, and could the harvest close it?** Three separate
answers, because there are three separate things affected.

**1. The lender features: no hole to handle.** Every column built in this notebook exists for
June 2025 and is exact, for the reason above. Nothing to fix.

**2. The bulk-file deltas: still holed, and the damage is one month per lag.** The spine
trick in `panel.py` turns June 2025 into an explicit all-NULL row, so `LAG(N)` is exactly `N`
calendar months back and reaching over the hole yields NULL rather than a wrong number. The
cost is that `d_*_3m` is NULL for the whole of 2025-09, `d_*_6m` for 2025-12, and `d_*_12m`
for 2026-06 (measured: 100% NULL in exactly those months, 3.7% / 7.0% / 13.9% otherwise, which
is the ordinary rate for companies that were not on the register a year earlier). **None of
those three months is a quarterly origin**, since the grid is Jan/Apr/Jul/Oct, so no training
row and no scoring row currently sits on a fully-NULL delta. The hole is real and it costs us
nothing where it lands.

**3. Could the harvested charge data fill it? For the charge half of the features, yes,
completely.** Twelve of the 41 features are functions of charge counts only:
`Mortgages.NumMortCharges`, `.NumMortOutstanding`, `.NumMortSatisfied`, the three
`d_charges_*`, the three `d_outstanding_*`, `d_satisfied_12m`, `new_charge_events_12m` and
`months_since_last_new_charge`. All twelve can be recomputed from `created_on` and
`satisfied_on` on a full calendar spine, exactly as this notebook already does for the lender
columns, which makes them gap-free rather than lag-based. Coverage is not a problem either:
the harvest targeted every company that ever showed `NumMortCharges > 0` in any snapshot, so a
company not in it has had zero charges in all 33 months and its charge deltas are zero by
construction, not unknown.

**What the harvest cannot fill is the other 29 features.** Accounts overdue and stale streaks,
`CompanyStatus` and everything derived from it, SIC/name/postcode changes, `debt_ratio` and
segment moves all come from the register snapshot itself. Reconstructing those for June 2025
would need the filing-history and company-profile endpoints across 1.37M active companies, not
158k, which at 2 requests per second is roughly eleven days of harvesting rather than one.

So my recommendation is: **not now, and it is written down as an option rather than done.**
Rebuilding the charge deltas from event dates would change twelve columns of the baseline
matrix that notebook 16's saved results were measured on, so it is another A/B tag and another
comparison, and it would be bought at the price of confounding it with the lender-feature
comparison that is already queued. It buys three non-origin months. If the origin grid ever
moves off Jan/Apr/Jul/Oct, or the panel is extended and the hole starts landing on a real
origin, this is the fix and the data for it is already on disk.

In [13]:
gap = con.execute(f"""
    SELECT snapshot_date,
           count(*)                             AS companies,
           avg(is_lbg_client::INT)              AS lbg_client_share,
           avg(competitor_entered_12m::INT)     AS competitor_entered_12m,
           avg(lbg_charge_satisfied_6m::INT)    AS lbg_satisfied_6m
    FROM read_parquet('{LENDER_GLOB}')
    WHERE snapshot_date BETWEEN DATE '2025-04-01' AND DATE '2025-08-01'
    GROUP BY 1 ORDER BY 1
""").df()

june = gap[gap["snapshot_date"] == pd.Timestamp("2025-06-01")]
assert len(june) == 1, "June 2025 missing from the lender panel"
neighbours = gap[gap["snapshot_date"].isin([pd.Timestamp("2025-05-01"), pd.Timestamp("2025-07-01")])]
assert (neighbours["lbg_client_share"].min() <= june["lbg_client_share"].iloc[0]
        <= neighbours["lbg_client_share"].max()), "June 2025 is not between its neighbours"
print("PASS: June 2025 exists in the lender panel and sits between May and July on every")
print("      measure. It is built from event dates, so the missing snapshot cannot reach it.")
gap

PASS: June 2025 exists in the lender panel and sits between May and July on every
      measure. It is built from event dates, so the missing snapshot cannot reach it.


,snapshot_date,companies,lbg_client_share,competitor_entered_12m,lbg_satisfied_6m
0,2025-04-01,152041,0.085924,0.001276,0.004019
1,2025-05-01,152448,0.085642,0.001325,0.003844
2,2025-06-01,152838,0.085299,0.001302,0.003697
3,2025-07-01,153333,0.084757,0.001350,0.003874
4,2025-08-01,153843,0.084339,0.001398,0.004004


### Check 7: leakage

*Rewritten 7 August 2026. What stood here was a passing test that could not detect the
bug it was written to catch, so the original wording is quoted rather than deleted.*

The original called itself **"the strongest test I could think of"**. It took the flat
charge table, deleted every charge *created* after 1 Jan 2025, blanked every satisfaction
dated after it, rebuilt the whole lender panel from that truncated history and asserted
the 2025-01 partition came back identical to the full one. It did. Every time.

It had to. The panel admitted a charge at `t` on `created_on <= t`, and the test
truncated the history on `created_on <= t`. Same clock on both sides of the equation, so
the answer was fixed before the data arrived. The question it actually asked was *"does
the panel use charges created after `t`"* (no, by construction). The question that
mattered was **"was a charge created before `t` knowable by `t`"**, and it never asked
it. **A blind-replay test is only as good as the clock you replay against.**

So the rewrite keeps two clocks apart and never lets them be the same object:

| | | |
|---|---|---|
| **reference clock** | what actually goes into the truncated history | `GREATEST(created_on, delivered_on) + REGISTRATION_LAG_DAYS <= source_date(t)` |
| **gate clock** | the admission rule of the panel under test, reconstructed so the rebuild differs in *history* only | whatever that panel was built with |

Two things changed and both matter. `created_on` became `visible_on`, the same expression
`LENDER_PANEL_SQL` gates on, because delivery and registration are what put a charge in
front of us, not creation. And the right-hand side became `source_date`, the date the
month's bulk extract was **actually** taken, which `panel.py` has been writing into every
panel partition all along, rather than the nominal 1st of the month.

The control assert is what makes the whole thing honest: I rebuild from the **full**
history under the reconstructed gate first and require that to come back identical to the
panel on disk. If it does, then any difference in the blind replay is attributable to the
truncated history and to nothing else, which is the confound that would otherwise let me
claim a leak when all I had done was change the code path.

In [21]:
import shutil
import numpy as np
import tempfile


def source_date_of(month: pd.Timestamp) -> pd.Timestamp:
    """The date the bulk extract for `month` was actually taken.

    Already on disk: `panel.py` writes `source_date` into every panel partition and
    documents it as "what point-in-time features are computed against". It is the
    same 33 dates as `ch_bulk.MANIFEST`.
    """
    part = (panel.PANEL_DIR / f"snapshot_date={month:%Y-%m-%d}" / "*.parquet").as_posix()
    return pd.Timestamp(
        con.execute(f"SELECT any_value(source_date) FROM read_parquet('{part}')").fetchone()[0]
    )


def visible_on(df: pd.DataFrame, lag_days: int) -> pd.Series:
    """The panel gate's own clock, rewritten in pandas.

    Identical to the SQL in `LENDER_PANEL_SQL`:
    `GREATEST(created_on, COALESCE(delivered_on, created_on)) + lag_days`.
    """
    delivered = df["delivered_on"].fillna(df["created_on"])
    return np.maximum(df["created_on"], delivered) + pd.Timedelta(days=lag_days)


def blind_replay(panel_dir, month, *, gate_lag_days, gate_uses_delivered,
                 ref_lag_days=charges.REGISTRATION_LAG_DAYS):
    """Rebuild the lender panel from only what the month-`month` extract could see,
    and diff that month's partition against the panel already on disk.

    Two clocks, and keeping them apart is the whole point:

    * the **reference clock** decides what goes into the truncated history. It is
      `visible_on(ref_lag_days) <= source_date(month)`: registered, by the day the
      register was actually read. It does not care what the panel believes.
    * the **gate clock** (`gate_lag_days`, `gate_uses_delivered`) reconstructs the
      admission rule of the panel being tested, so the rebuild differs from it in
      the *history* only and not in the *code path*. The control assert below
      proves that reconstruction is exact.
    """
    src = source_date_of(month)
    known = visible_on(flat, ref_lag_days) <= src
    truncated = flat[known].copy()
    truncated.loc[truncated["satisfied_on"] > src, "satisfied_on"] = pd.NaT

    print(f"month {month:%Y-%m}, bulk extract actually taken {src:%Y-%m-%d}, "
          f"reference lag {ref_lag_days}d")
    print(f"{len(flat):,} charge-lender rows -> {len(truncated):,} in the register at that "
          f"extract ({len(flat) - len(truncated):,} dropped)")
    naive = flat["created_on"] <= month
    print(f"the old test's clock (created_on <= {month:%Y-%m-%d}) would have kept "
          f"{int(naive.sum()):,}, i.e. {int((naive & ~known).sum()):,} charges it called "
          f"knowable that the register could not yet see")

    tmp = Path(tempfile.mkdtemp())
    part = f"snapshot_date={month:%Y-%m-%d}/*.parquet"
    try:
        def build(df, out):
            d = df.copy()
            if not gate_uses_delivered:
                d["delivered_on"] = d["created_on"]     # visible_on collapses to created_on
            d.to_parquet(tmp / "f.parquet", index=False)
            charges.build_lender_panel(flat_path=tmp / "f.parquet", out_dir=out,
                                       threads=6, lag_days=gate_lag_days)
            return con.execute(f"SELECT * FROM read_parquet('{(out / part).as_posix()}') "
                               'ORDER BY "CompanyNumber"').df()

        control = build(flat, tmp / "control")
        blind = build(truncated, tmp / "blind")
    finally:
        shutil.rmtree(tmp)

    on_disk = con.execute(f"SELECT * FROM read_parquet('{(Path(panel_dir) / part).as_posix()}') "
                          'ORDER BY "CompanyNumber"').df()
    cols = [c for c in on_disk.columns if c not in ("snapshot_date", "CompanyNumber")]
    control_ok = (control[["CompanyNumber"] + cols].reset_index(drop=True)
                  .equals(on_disk[["CompanyNumber"] + cols].reset_index(drop=True)))
    print(f"control (full history, reconstructed gate) reproduces the panel on disk: {control_ok}")

    a = on_disk.set_index("CompanyNumber")[cols]
    b = blind.set_index("CompanyNumber")[cols]
    j = a.join(b, how="outer", lsuffix="_disk", rsuffix="_blind")
    ne = pd.DataFrame({c: ~((j[f"{c}_disk"] == j[f"{c}_blind"])
                            | (j[f"{c}_disk"].isna() & j[f"{c}_blind"].isna()))
                       for c in cols})
    per_col = ne.sum().sort_values(ascending=False)
    n_diff = int(ne.any(axis=1).sum())
    print(f"\non disk {len(a):,} companies, blind replay {len(b):,}, union {len(j):,}")
    print(f"{n_diff:,} companies differ on at least one column ({n_diff / len(j):.2%})")
    if n_diff:
        print(per_col[per_col > 0].to_string())
    return dict(identical=(n_diff == 0), n_diff=n_diff, n_union=len(j),
                control_ok=control_ok, per_col=per_col)

**The half that has to fail.** Against `data/processed/lender_panel/`, the un-lagged panel
this notebook originally built, the rewritten test must fail, because that panel really
does gate on `created_on` against the nominal 1st. A test that has never failed has never
been tested, and this is the assertion the whole rewrite exists for.

In [22]:
v7_old = blind_replay(charges.LENDER_PANEL_DIR, pd.Timestamp("2025-01-01"),
                      gate_lag_days=0, gate_uses_delivered=False)

assert v7_old["control_ok"], "the reconstructed gate does not reproduce the panel on disk"
assert not v7_old["identical"], \
    "verification 7 still passes on the un-lagged panel, so the rewrite did not move the clock"
print("\nFAILS, which is the point. The un-lagged panel at 2025-01 reports charge activity")
print("that the register could not see until after that month's extract was taken.")

month 2025-01, bulk extract actually taken 2025-01-01, reference lag 21d
572,776 charge-lender rows -> 540,030 in the register at that extract (32,746 dropped)
the old test's clock (created_on <= 2025-01-01) would have kept 541,485, i.e. 1,455 charges it called knowable that the register could not yet see
control (full history, reconstructed gate) reproduces the panel on disk: True

on disk 150,695 companies, blind replay 150,360, union 150,695
1,164 companies differ on at least one column (0.77%)
n_charges_outstanding                   1163
competitor_charge_created_6m             794
n_distinct_lenders                       641
n_competitor_lenders                     631
lbg_share_of_outstanding                 599
primary_lender_group                     485
n_lbg_charges_outstanding                390
competitor_entered_12m                   347
is_lbg_client                            346
ever_lbg_client                          344
lbg_charge_satisfied_6m                  335
mo

**And the half that has to pass, so I know it is a test and not a tripwire.** Same test,
same month, same reference clock, pointed at `lender_panel_asof21/`, the panel the recorded
lender runs were actually trained on. That gate is `visible_on(21d) <= snapshot_date`, which
is at least as strict as the reference clock, so it should come back clean. If the rewritten
check failed on everything it would be worthless.

In [23]:
v7_asof21 = blind_replay(Path("data/processed/lender_panel_asof21"),
                         pd.Timestamp("2025-01-01"),
                         gate_lag_days=charges.REGISTRATION_LAG_DAYS,
                         gate_uses_delivered=True)

assert v7_asof21["control_ok"], "the reconstructed gate does not reproduce the panel on disk"
assert v7_asof21["identical"], "the 21-day panel reaches forward at 2025-01"
print("\nPASSES. Same test, same month, same reference clock, opposite verdict, so the")
print("test discriminates between the two panels rather than simply failing on everything.")

month 2025-01, bulk extract actually taken 2025-01-01, reference lag 21d
572,776 charge-lender rows -> 540,030 in the register at that extract (32,746 dropped)
the old test's clock (created_on <= 2025-01-01) would have kept 541,485, i.e. 1,455 charges it called knowable that the register could not yet see
control (full history, reconstructed gate) reproduces the panel on disk: True

on disk 150,360 companies, blind replay 150,360, union 150,360
0 companies differ on at least one column (0.00%)

PASSES. Same test, same month, same reference clock, opposite verdict, so the
test discriminates between the two panels rather than simply failing on everything.


**One month is not enough, and the month I picked was the wrong one.** 2025-01 was the
original test's month, so I kept it for continuity, but `source_date(2025-01)` is
2025-01-01: the extract really was cut on the nominal 1st that month. That month therefore
exercises only *half* the fix, the `created_on` to `visible_on` half, and says nothing at
all about the `source_date` half. A test that only ever exercises half its clock is the
same class of gap as the one I just spent this section fixing, so here is the other half.

`ch_bulk.MANIFEST` has four months where the extract was not taken on the 1st:
2023-10-04, 2023-12-04, **2024-02-07** and 2026-03-02. February 2024 is the largest of
them, six days off the nominal date, so it is the month that isolates the extract-date
half. Three clocks, both months:

In [25]:
def clock_counts(month):
    """The three clocks side by side, in charges admitted as knowable at `month`.

    C0 is what the original test used, C1 is half the fix (visible_on instead of
    created_on) and C2 is the whole of it (and against the real extract date).
    """
    src = source_date_of(month)
    vis = visible_on(flat, charges.REGISTRATION_LAG_DAYS)
    c0 = flat["created_on"] <= month                    # the old test's clock
    c1 = vis <= month                                   # half A only: visible_on, nominal 1st
    c2 = vis <= src                                     # both halves: visible_on, real extract
    return {
        "month": f"{month:%Y-%m}",
        "source_date": f"{src:%Y-%m-%d}",
        "drift_days": (src - month).days,
        "C0 created_on <= 1st": int(c0.sum()),
        "C1 visible_on <= 1st": int(c1.sum()),
        "C2 visible_on <= source_date": int(c2.sum()),
        "half A removes (C0 & ~C1)": int((c0 & ~c1).sum()),
        "half B restores (C2 & ~C1)": int((c2 & ~c1).sum()),
        "net wrongly admitted (C0 & ~C2)": int((c0 & ~c2).sum()),
    }

clocks = pd.DataFrame([clock_counts(pd.Timestamp(m))
                       for m in ("2025-01-01", "2024-02-01")]).set_index("month").T
clocks

month,2025-01,2024-02
source_date,2025-01-01,2024-02-07
drift_days,0,6
C0 created_on <= 1st,541485,522711
C1 visible_on <= 1st,540030,521468
C2 visible_on <= source_date,540030,521675
half A removes (C0 & ~C1),1455,1243
half B restores (C2 & ~C1),0,207
net wrongly admitted (C0 & ~C2),1455,1036


**The two halves pull in opposite directions, and that is the whole argument for
decomposing the 21 days rather than tuning them.**

At 2024-02 the `visible_on` half removes 1,243 charges that `created_on` called knowable,
and the `source_date` half hands 207 of them straight back, because those charges *were*
in the file, which was cut on the 7th rather than the 1st. Net, the old clock wrongly
admitted 1,036. At 2025-01, where there is no drift, the same two numbers are 1,455 and
zero, and the net is the full 1,455.

So roughly a sixth of the correction at a drift month is not processing lag at all, it is
the extract date, and it is a quantity that was sitting in `panel.py`'s `source_date`
column the entire time. A single flat constant cannot represent it: 21 days is 21 days in
every month, but the real gap between the nominal 1st and the file is 0 days in 29 months
and 1 to 6 days in the other four. Fitting one number to cover both a thing we know
exactly and a thing we have to estimate is how the constant ended up overshooting, and it
is exactly what stream D is pulling apart.

In [26]:
v7_old_feb = blind_replay(charges.LENDER_PANEL_DIR, pd.Timestamp("2024-02-01"),
                          gate_lag_days=0, gate_uses_delivered=False)

assert v7_old_feb["control_ok"], "the reconstructed gate does not reproduce the panel on disk"
assert not v7_old_feb["identical"], "the un-lagged panel replays clean at 2024-02, which it must not"
print("\nFAILS at the drift month too, and note this is the *harder* month for the test:")
print("the reference clock is six days more generous here, so the truncated history is")
print("larger and the test has less room to find a difference. It finds one anyway.")

month 2024-02, bulk extract actually taken 2024-02-07, reference lag 21d
572,776 charge-lender rows -> 521,675 in the register at that extract (51,101 dropped)
the old test's clock (created_on <= 2024-02-01) would have kept 522,711, i.e. 1,036 charges it called knowable that the register could not yet see
control (full history, reconstructed gate) reproduces the panel on disk: True

on disk 145,804 companies, blind replay 145,550, union 145,804
853 companies differ on at least one column (0.59%)
n_charges_outstanding                   853
competitor_charge_created_6m            602
n_distinct_lenders                      437
lbg_share_of_outstanding                426
n_competitor_lenders                    425
primary_lender_group                    323
n_lbg_charges_outstanding               282
is_lbg_client                           267
competitor_entered_12m                  267
ever_lbg_client                         264
lbg_charge_satisfied_6m                 254
months_since_la

And the same month against the 21-day panel, which is the pointed version of the same
observation.

In [27]:
v7_asof21_feb = blind_replay(Path("data/processed/lender_panel_asof21"),
                             pd.Timestamp("2024-02-01"),
                             gate_lag_days=charges.REGISTRATION_LAG_DAYS,
                             gate_uses_delivered=True)

assert v7_asof21_feb["control_ok"], "the reconstructed gate does not reproduce the panel on disk"
assert v7_asof21_feb["identical"], "the 21-day panel reaches forward at 2024-02"
n_stale = clocks.loc["half B restores (C2 & ~C1)", "2024-02"]
print(f"\nPASSES, but read it precisely: it passes because it is *stricter* than it needs")
print(f"to be. {n_stale:,} charges were in the 2024-02 extract and the 21-day gate refuses")
print("them, because it measures against the nominal 1st and the file was cut on the 7th.")
print("Safe, and six days stale. That is the gap stream D is closing.")

month 2024-02, bulk extract actually taken 2024-02-07, reference lag 21d
572,776 charge-lender rows -> 521,675 in the register at that extract (51,101 dropped)
the old test's clock (created_on <= 2024-02-01) would have kept 522,711, i.e. 1,036 charges it called knowable that the register could not yet see
control (full history, reconstructed gate) reproduces the panel on disk: True

on disk 145,496 companies, blind replay 145,496, union 145,496
0 companies differ on at least one column (0.00%)

PASSES, but read it precisely: it passes because it is *stricter* than it needs
to be. 207 charges were in the 2024-02 extract and the 21-day gate refuses
them, because it measures against the nominal 1st and the file was cut on the 7th.
Safe, and six days stale. That is the gap stream D is closing.


**The half that is still pending.** `lender_panel_asof21/` passes both months by being
stricter than it needs to be, and at 2024-02 I can now put a number on the cost of that:
207 charges that the register could see and the gate would not. Stream D is rebuilding the
gate to measure against `source_date` directly and to calibrate the residual processing lag
rather than assume it, which is a *tighter* gate, not a looser one, and tighter is exactly
where this test earns its keep. The cell below is that panel's acceptance test, written now
and run the moment the directory exists, on both months.


In [28]:
CALIB_DIR = Path("data/processed/lender_panel_calib")

if not CALIB_DIR.exists():
    print(f"{CALIB_DIR} not on disk yet. This is stream D's rebuild: the gate becomes")
    print("visible_on(calibrated lag) <= source_date instead of the nominal 1st of the")
    print("month, and charges.REGISTRATION_LAG_DAYS becomes the calibrated residual.")
    print("The assertion below is the acceptance test for that rebuild; when the panel")
    print("lands, re-run this cell and nothing else in it changes.")
else:
    for month in ("2025-01-01", "2024-02-01"):
        r = blind_replay(CALIB_DIR, pd.Timestamp(month),
                         gate_lag_days=charges.REGISTRATION_LAG_DAYS,
                         gate_uses_delivered=True)
        assert r["control_ok"], "the reconstructed gate does not reproduce the panel on disk"
        assert r["identical"], f"at {month} a lender feature still uses a charge invisible at t"
        print()
    print("PASS: the calibrated panel replays blind against the real extract date, at a")
    print("month with no drift and at the month with the most.")

data/processed/lender_panel_calib not on disk yet. This is stream D's rebuild: the gate becomes
visible_on(calibrated lag) <= source_date instead of the nominal 1st of the
month, and charges.REGISTRATION_LAG_DAYS becomes the calibrated residual.
The assertion below is the acceptance test for that rebuild; when the panel
lands, re-run this cell and nothing else in it changes.


That is checks 1, 2, 3, 4, 5 and 7 done. Check 6 is the switching base rate, which needs the
label, so it comes after the integration.

## 5. Integration

Two things to wire up: the lender features onto the existing matrices, and the fifth target.

**I deliberately did not overwrite the step 6 matrices.** `data/processed/model_matrix/` is
what notebook 16 trained on, and the `baseline` run under `reports/runs/` records the exact
feature list behind every
number in it. Widening those rows in place would silently invalidate that report and, worse,
destroy the only baseline I have to measure whether a day of harvesting bought anything. So
the lender build writes to `model_matrix_lender/` beside it, which is the same A/B shape as
the strict-versus-extended contract comparison in notebook 14a. Two directories, one switch,
both reproducible.

Same reasoning for the code: `targets.FEATURE_COLS` is untouched and
`targets.FEATURE_COLS_LENDER` is the wider list, and `switching` lives in
`targets.LENDER_TARGETS` rather than being appended to `targets.TARGETS`, so notebooks 15 and
16 still loop over exactly the four targets whose matrices are on disk.

In [15]:
%%time
summaries = []
for name in list(targets.TARGETS) + ["switching"]:
    kwargs = {"label_dir": targets.SWITCHING_LABEL_DIR} if name == "switching" else {}
    # The switching label is built below; this reuses the version already on disk.
    if name == "switching" and not targets.SWITCHING_LABEL_DIR.exists():
        targets.build_switching_labels(threads=6)
    s = targets.build_matrix(name, lender_dir=charges.LENDER_PANEL_DIR,
                             out_dir=targets.LENDER_MATRIX_DIR, threads=6, **kwargs)
    summaries.append(s)
    print(f"{name:<15} {s['rows'].sum():>10,} rows  {int(s['positives'].sum()):>9,} positives",
          flush=True)

lender_summary = pd.concat(summaries, ignore_index=True)
print(f"\n{len(targets.FEATURE_COLS)} features -> {len(targets.FEATURE_COLS_LENDER)} with the "
      f"lender columns (+{len(targets.LENDER_FEATURE_COLS)})")

lending            457,941 rows     41,788 positives


insolvency         520,142 rows     47,262 positives


voluntary_exit  11,917,730 rows  1,083,664 positives


growth           1,657,611 rows    150,806 positives


switching            9,085 rows        797 positives



41 features -> 54 with the lender columns (+13)
CPU times: user 1min 21s, sys: 12.2 s, total: 1min 34s
Wall time: 19.1 s


Row counts and positive counts are identical to notebook 15's for the four original targets,
which is the point: the join is a `LEFT JOIN` that adds columns and cannot add or drop rows.

Now, how much of the population actually gets lender information? Most companies have never
held a charge, and for them a zero is the *correct* answer rather than a missing one: they are
not LBG clients, they have no lenders, and coalescing to 0 says so. `months_since_*` stays
NULL, because "never borrowed from Lloyds" has no zero-equivalent. Same convention the
contract features already use.

In [16]:
lending_lender = pd.read_parquet(targets.LENDER_MATRIX_DIR / "lending")
fill = pd.DataFrame([
    {"feature": c,
     "non_null": lending_lender[c].notna().mean(),
     "non_zero / true": (lending_lender[c].fillna(0).astype(float) > 0).mean()
                        if lending_lender[c].dtype != object else None}
    for c in charges.LENDER_FEATURE_COLS
])
fill.style.format({"non_null": "{:.1%}", "non_zero / true": "{:.1%}"}, na_rep="")

,feature,non_null,non_zero / true
0,n_charges_outstanding,100.0%,12.1%
1,n_lbg_charges_outstanding,100.0%,1.4%
2,is_lbg_client,100.0%,1.4%
3,lbg_share_of_outstanding,12.1%,1.4%
4,n_distinct_lenders,100.0%,10.3%
5,n_competitor_lenders,100.0%,9.2%
6,months_since_last_lbg_charge_created,3.0%,3.0%
7,months_since_last_lbg_satisfaction,2.0%,2.0%
8,competitor_entered_12m,100.0%,0.1%
9,lbg_charge_satisfied_6m,100.0%,0.1%


12.1% of the lending matrix carries an outstanding charge, 9.2% has a competitor lender and
1.4% is a current LBG client (3.0% has ever been one). Those are higher than the 7.8%
population rate because the matrix downsamples negatives, and companies that take charges are
exactly the ones the lending label fires on. The only column that is genuinely NULL rather
than zero is `months_since_last_lbg_charge_created`, at 3.0% populated, which is precisely the
set that has ever borrowed from us.

### The fifth label: switching

This is the client's actual "attrition", and it is the only label here the bulk file could
never have produced. It is built by `targets.build_switching_labels`, and the SQL behind it is
`targets.SWITCHING_SQL` if anyone wants to read the exact version.

It is computed in four steps, all of them standing at an origin month `t` and looking forward
six months.

**Step 1, the base population.** Every (company, `t`) where the lender panel says
`n_lbg_charges_outstanding > 0` *and* the delta panel says the company is active at `t`. You
cannot leave a bank you do not use, so non-clients are not in the denominator at all, and the
activity join means a company that simply dissolved is not counted as having switched. That
base is about 12,000 companies per origin, which is why the label has 121,655 rows rather than
the millions the other four have.

**Step 2, did we lose the whole relationship?** Look up the same company in the lender panel
at `t+6m`. The condition is `n_lbg_charges_outstanding = 0`: not fewer, none. If the panel has
no row at `t+6m` (the window runs off the end of the panel) the label is **NULL, not 0**,
because "I cannot see yet" and "it did not happen" are different statements and only one of
them belongs in a training set.

**Step 3, was it an event?** At least one LBG charge must have a `satisfied_on` strictly after
`t` and on or before `t+6m`. This is what stops a company that was already drifting out of the
data being scored as a departure: the release has to happen inside the window I am claiming to
predict.

**Step 4, was it replaced?** For at least one of those satisfactions, a **competitor** charge
must have been created within three months either side of it. This is the clause that
separates switching from **deleveraging**. A company that repays its Lloyds loan and borrows
nothing has not gone anywhere, and putting it on a win-back list wastes an RM's afternoon.
`lbg_retail` and `unclassified` count as neither ours nor theirs, so a Halifax charge or a
director's personal loan cannot fake a defection.

A worked example, taking `t = 2025-01` and `t+6m = 2025-07`:

| Company | LBG outstanding at `t` | LBG outstanding at `t+6m` | LBG satisfaction in window | Competitor charge within 3m of it | `y_switching` |
|---|---|---|---|---|---|
| A | 2 | 0 | yes, 2025-03 | yes, Barclays 2025-04 | **1** |
| B | 2 | 0 | yes, 2025-03 | no | 0 (deleveraged) |
| C | 3 | 1 | yes, 2025-05 | yes, HSBC 2025-05 | 0 (partial, see below) |
| D | 1 | 1 | no | Santander 2025-02 | 0 (additional borrowing) |
| E | 1 | not in the panel at `t+6m` | - | - | **NULL** |
| F | 0 | - | - | - | not in the population |

Company C is the interesting row and it is the definition's main cost: they moved two thirds
of their book to HSBC and this label calls it a non-event. That is a deliberate choice for a
first pass (a full exit is unambiguous, a partial one shades into ordinary refinancing) but it
is exactly the loosening I cost out in section 7.

In [17]:
n_switch = targets.build_switching_labels(threads=6)
switch_rates = targets.check_base_rates(targets.SWITCHING_LABEL_DIR, targets.LENDER_TARGETS)
positives = int((switch_rates["n"] * switch_rates["base_rate"]).sum())
print(f"{n_switch:,} label rows, {positives:,} positive switching events across "
      f"{len(switch_rates)} quarterly origins")
switch_rates.style.format({"n": "{:,}", "base_rate": "{:.3%}"})

121,655 label rows, 797 positive switching events across 10 quarterly origins


,target,origin_month,n,base_rate
0,switching,2023-10-01 00:00:00,"12,939",0.595%
1,switching,2024-01-01 00:00:00,"12,740",0.628%
2,switching,2024-04-01 00:00:00,"12,564",0.653%
3,switching,2024-07-01 00:00:00,"12,397",0.734%
4,switching,2024-10-01 00:00:00,"12,219",0.638%
5,switching,2025-01-01 00:00:00,"12,052",0.531%
6,switching,2025-04-01 00:00:00,"11,927",0.570%
7,switching,2025-07-01 00:00:00,"11,791",0.619%
8,switching,2025-10-01 00:00:00,"11,651",0.790%
9,switching,2026-01-01 00:00:00,"11,375",0.809%


### Check 6: the base rate

**0.53% to 0.81% per quarter, so well under the 2% the plan set as the line where the
definition is wrong.** That is check 6 passed. It also lands almost exactly where the scoping
estimate put it: around 1,000 to 2,000 events over 33 months, and I measure 797 at quarterly
origins over a base population of about 12,000 LBG clients per origin.

And that number triggers the stop rule I wrote into the plan before I had it:

> **Stretch only.** If the positive class lands under ~1,000 usable events at quarterly
> origins, ship the rule-based switch feed plus a competitor-encroachment ranking and say so,
> rather than a model whose precision@500 cannot be defended.

797 is under 1,000. Split out of time, the test period would hold something like 150 positives,
and precision@500 on 150 positives is a number with an enormous confidence interval attached.
So **I am not training a fifth model**, and that is a decision made against a threshold set in
advance rather than after seeing which answer looked better.

What I ship instead is below. The matrix is built and on disk either way, so if the team later
wants the model, or wants to widen the definition to partial exits, nothing has to be rebuilt.

## 6. What this actually delivers to the client

Three things, none of which needed a model.

**1. The switch feed (Attrition).** Companies that were LBG clients and have just left, with
who they left for. This is a list an RM can work today, and it is retrospective by
construction: it reports what happened, so it does not need a model to be trustworthy.

**2. Competitor encroachment (Attrition, early warning).** Companies that were LBG-only a year
ago and have taken a charge from someone else since. Wallet-share erosion precedes full
defection, so this is the list that gives the RM lead time rather than a post-mortem.

**3. Borrowing elsewhere (Growth).** Companies with an outstanding competitor charge and no LBG
charge at all: already borrowing, demonstrably fundable, and not ours. That is a far sharper
prospect list than a generic lending-readiness score, and it is a filter rather than a model.

In [18]:
LIVE = pd.Timestamp(panel.LAST_MONTH)
live_glob = (charges.LENDER_PANEL_DIR / f"snapshot_date={LIVE:%Y-%m-%d}" / "*.parquet").as_posix()
delta_live = (panel.DELTA_DIR / f"snapshot_date={LIVE:%Y-%m-%d}" / "*.parquet").as_posix()

feeds = con.execute(f"""
    WITH live AS (
        SELECT l.*, p."CompanyName", p.sector, p.segment
        FROM read_parquet('{live_glob}') l
        JOIN read_parquet('{delta_live}') p USING ("CompanyNumber")
        WHERE p.is_active
    )
    SELECT 'lbg_client'          AS feed, count(*) AS companies FROM live WHERE is_lbg_client
    UNION ALL
    SELECT 'just_switched',      count(*) FROM live
        WHERE NOT is_lbg_client AND ever_lbg_client
          AND lbg_charge_satisfied_6m AND competitor_charge_created_6m
    UNION ALL
    SELECT 'competitor_entered', count(*) FROM live WHERE is_lbg_client AND competitor_entered_12m
    UNION ALL
    SELECT 'borrowing_elsewhere', count(*) FROM live
        WHERE NOT is_lbg_client AND n_competitor_lenders > 0
""").df()
feeds

,feed,companies
0,lbg_client,11003
1,just_switched,75
2,competitor_entered,115
3,borrowing_elsewhere,70030


In [19]:
switch_feed = con.execute(f"""
    SELECT l."CompanyNumber", p."CompanyName", p.sector, p.segment,
           l.primary_lender_group AS now_banks_with,
           l.n_charges_outstanding, l.months_since_last_lbg_satisfaction
    FROM read_parquet('{live_glob}') l
    JOIN read_parquet('{delta_live}') p USING ("CompanyNumber")
    WHERE p.is_active
      AND NOT l.is_lbg_client AND l.ever_lbg_client
      AND l.lbg_charge_satisfied_6m AND l.competitor_charge_created_6m
    ORDER BY l.n_charges_outstanding DESC, l.months_since_last_lbg_satisfaction
""").df()

out = Path("data/processed/scores") / f"lender_feeds_{LIVE:%Y-%m}.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
switch_feed.to_parquet(out, index=False)
print(f"{len(switch_feed):,} companies left LBG for a competitor in the last six months "
      f"-> {out}")
switch_feed.head(10)

75 companies left LBG for a competitor in the last six months -> data/processed/scores/lender_feeds_2026-07.parquet


,CompanyNumber,CompanyName,sector,segment,now_banks_with,n_charges_outstanding,months_since_last_lbg_satisfaction
0,11414518,MEASURED IDENTITY HUB LIMITED,"Technology, legal & professional",Large,other_bank,9,1
1,07928622,AEROCO GROUP INTERNATIONAL LIMITED,Manufacturing,Subsidiary,challenger_bank,6,4
2,01524697,BLAZE SIGNS LIMITED,Manufacturing,Large,asset_invoice_finance,6,4
3,02926711,CAMBORNE JOINERY LIMITED,Manufacturing,Small,hsbc,5,1
4,05410302,ORIGIN COFFEE LTD,Manufacturing,Large,hsbc,5,3
5,04959428,SERVICED DISPENSE EQUIPMENT (HOLDINGS) LIMITED,"Technology, legal & professional",Large,hsbc,3,1
6,09835011,CAPITAL BELTING LIMITED,Manufacturing,Small,hsbc,3,1
7,07361465,E S ENGINEERING SERVICES LIMITED,NaN,Small,natwest,3,2
8,07826616,BSP ENGINEERING SERVICES (UK) LTD,"Technology, legal & professional",Small,challenger_bank,3,3
9,05231232,SOHONET AUSTRALIA LIMITED,"Technology, legal & professional",Dormant,trustee_spv,3,4


In [20]:
encroachment = con.execute(f"""
    SELECT l."CompanyNumber", p."CompanyName", p.sector, p.segment,
           l.n_lbg_charges_outstanding, l.n_competitor_lenders,
           l.lbg_share_of_outstanding, l.primary_lender_group
    FROM read_parquet('{live_glob}') l
    JOIN read_parquet('{delta_live}') p USING ("CompanyNumber")
    WHERE p.is_active AND l.is_lbg_client AND l.competitor_entered_12m
    ORDER BY l.lbg_share_of_outstanding, l.n_competitor_lenders DESC
""").df()
print(f"{len(encroachment):,} current LBG clients took a competitor charge in the last 12 months")
encroachment.head(10)

115 current LBG clients took a competitor charge in the last 12 months


,CompanyNumber,CompanyName,sector,segment,n_lbg_charges_outstanding,n_competitor_lenders,lbg_share_of_outstanding,primary_lender_group
0,SC183918,JAMES DONALDSON TIMBER LIMITED,Manufacturing,Large,1,2,0.200000,challenger_bank
1,SC099182,DONALDSON TIMBER ENGINEERING LIMITED,Manufacturing,Large,1,2,0.200000,challenger_bank
2,SC311961,DONALDSON DOOR SYSTEMS LIMITED,Manufacturing,Subsidiary,1,2,0.200000,challenger_bank
3,03745935,CHAMBERTIN CAPITAL (UK) LIMITED,"Technology, legal & professional",Large,1,1,0.200000,trustee_spv
4,04016333,RUMENCO LIMITED,Manufacturing,Large,1,1,0.250000,hsbc
5,SC408719,CURLE STEWART LIMITED,"Technology, legal & professional",Small,1,1,0.333333,challenger_bank
6,13302690,EMS GROUP SOLUTIONS LTD,"Technology, legal & professional",Large,1,1,0.333333,challenger_bank
7,06692101,HOSE SOLUTIONS LIMITED,Manufacturing,Small,1,1,0.333333,challenger_bank
8,11756687,STEEL DYNAMICS EXPRESS (NW) LIMITED,"Technology, legal & professional",Small,1,1,0.333333,challenger_bank
9,01391511,EXCEL PRECISION (WIRE SPARK EROSION) LIMITED,Manufacturing,Small,1,1,0.333333,hsbc


## 7. The decision on `switching`, stated plainly

Two things I want on the record, because they are easy to confuse and the distinction is the
whole point of this notebook.

**`switching` is not going into the final model as a target. Confirmed.** 797 positives is
under the 1,000 the plan set in advance, so the fifth model is not trained. The label is built
and on disk (`data/processed/labels_switching/`, and the matrix at
`model_matrix_lender/switching/`) so the decision is reversible without rebuilding anything,
but as of today notebook 16 trains **four** targets and `targets.TARGETS` still holds exactly
those four. `switching` lives in `targets.LENDER_TARGETS`, deliberately separate, so nothing
picks it up by accident.

**The lender variables absolutely are going in, as features. Confirmed.** All thirteen of
them, on all four existing targets, in `model_matrix_lender/`. Nothing this notebook produced
is being predicted; it is all being used to predict. So:

| | Role | Where it goes |
|---|---|---|
| the 13 lender columns | **explanatory variables** (X) | joined onto all four target matrices, 41 features become 54 |
| `y_switching` | would-be target (y) | built, measured, **not trained**; kept for later |
| the three feeds | rule-based deliverable | `data/processed/scores/lender_feeds_2026-07.parquet`, no model involved |

That makes this notebook a sibling of 13 and 14 (feature builders) rather than of 15 (the
label builder). The only reason it touches a label at all is that the same charge data happens
to answer a question the bulk file cannot, and it was worth measuring how big that question is
before deciding not to model it.

### Do not train it, or widen the definition? The numbers

The alternative to dropping it is loosening the definition until the positive class is big
enough, so I measured what each loosening actually buys before arguing about it. All four rows
below use the same base population and the same replacement clause; only the exit test and the
horizon move. "Trainable" is what survives `train.split_origins`: the last two origins are the
out-of-time test, one more is eaten by the embargo, and `FIRST_FULL_ORIGIN` is dropped
automatically when honouring it would leave fewer than three training origins.

| Option | Exit test | H | Total positives | Train origins | Train positives | Test positives |
|---|---|---|---|---|---|---|
| **A: as built (today)** | LBG charges go to **zero** | 6m | 797 | 3 (2024-10 to 2025-04) | 210 | 184 |
| **B: widen to partial** | LBG charges **decrease** | 6m | 1,028 | 3 (2024-10 to 2025-04) | 273 | 252 |
| **C: lengthen the window** | goes to zero | 12m | 1,256 | 3 (2023-10 to 2024-04) | 490 | 328 |
| **D: both** | decreases | 12m | 1,556 | 3 (2023-10 to 2024-04) | 592 | 421 |

**What widening to partial exits costs (A to B).** It buys 231 positives, a 29% increase, and
it pays for them by admitting the ambiguous cases. A company going from 3 LBG charges to 2 has
usually just repaid one facility on schedule; the label would now call that attrition, and the
model would learn "charges get repaid", which is true of every borrower and useless. The
signal-to-noise of the positive class goes down at the same time as its size goes up, and 273
training positives against 54 features is still not a modelling proposition. This is the worst
of the four: it is the option that changes what the word "attrition" means to the client
without buying enough data to be worth it.

**What lengthening the horizon costs (A to C).** More positives per row, because six more
months is six more chances to leave, and the training rows land in the oldest origins. That is
the expensive part: a 12-month horizon means the last usable origin is 2025-07, the embargo
eats three origins instead of one, and the surviving training origins (2023-10 to 2024-04) sit
in the region where `FIRST_FULL_ORIGIN` says the 12-month deltas are only partly formed. It is
the same problem `growth` already has, and there I can at least see it in the AUC. Here I
would be diagnosing it on 490 positives.

**My recommendation is still A, not trained.** Even the widest option, D, gives 592 training
positives spread over three origins and 421 test positives, and the metric the client cares
about is precision@N on a call list. With 421 test positives across roughly 24,000 test rows,
a precision@500 estimate has a confidence interval wide enough to cover both "twice as good as
random" and "no better than random", and I would not put a number like that in front of a
relationship manager. The honest version of this deliverable is the retrospective feed above,
which is a list of companies that **did** leave, measured rather than predicted, and 75 of them
is a real afternoon's work for an RM.

**What would change my mind**, in order of how much it would help: another year of panel
history, which turns three training origins into seven and is the single biggest constraint
here; monthly rather than quarterly origins, which multiplies the rows by three at the cost of
heavily overlapping labels; or the definition moving from "charges" to "value secured", which
would let a partial exit be measured as a proportion of exposure rather than a count and would
make option B mean something. None of those is available this month, which is why the
recommendation is to wait rather than to weaken.

## 8. Where this leaves us

On disk now:

- `data/raw/charges/` - 32 NDJSON shards, 368 MB, 158,684 companies, 558,693 charges. This is
  the archive; nothing downstream ever needs to re-harvest, including for taxonomy changes.
- `data/processed/charges_flat.parquet` - charge-lender grain with the group classification.
- `data/processed/lender_panel/snapshot_date=*/` - 5.1M company-month rows, 12 features plus
  the primary lender group, joining exactly like `contracts_asof/`.
- `data/processed/labels_switching/` - the fifth label, 797 positives.
- `data/processed/model_matrix_lender/{lending,insolvency,voluntary_exit,growth,switching}/` -
  the wider matrices, built beside the originals rather than over them.
- `data/processed/scores/lender_feeds_2026-07.parquet` - the switch feed.

Verification summary:

| # | Check | Result |
|---|---|---|
| 1 | Harvest completeness | 158,684 / 158,684 companies, 0 x 404, 0 duplicates |
| 2 | Charge counts reconcile with the bulk file | 97.7% exact, 99.3% within one; the 2.2% asymmetry was the leak in plain sight, see the addendum in section 4 |
| 3 | Point-in-time flip | `is_lbg_client` flips the month after satisfaction |
| 4 | June 2025 | present and smooth; built from event dates, so no hole to step over |
| 5 | Taxonomy coverage | 77.5% classified, tail is individuals and SPVs, not banks |
| 6 | Switching base rate | 0.53% to 0.81% per quarter, under the 2% ceiling |
| 7 | Leakage | **rewritten 7 Aug 2026.** Blind replay against the real extract date, at two months. **Fails** on `lender_panel/`: 1,164 of 150,695 companies (0.77%) at 2025-01, 853 of 145,804 (0.59%) at 2024-02. Passes on `lender_panel_asof21/` at both. See below. |

**On check 7, plainly.** The version of this check that shipped with the notebook was a
passing test that could not fail. It truncated the charge history on `created_on <= t`,
which is the same clock the panel gated on, so it compared a rule against itself and
called the agreement evidence. Its markdown called it "the strongest test I could think
of", and it is the sentence in this notebook I would most like back. **A blind-replay test
is only as good as the clock you replay against.** What it tests now is whether a feature
at `t` uses a charge that was not yet in the register when that month's bulk extract was
taken, replayed against `source_date` rather than against the panel's own assumption. On
the panel it was written to protect, it fails: 1,164 companies at a single month, across
twelve of the thirteen lender columns. It is also the reason `lender_panel/` is kept on
disk rather than overwritten, since a test that fails needs something to fail against.

It runs at two months on purpose. 2025-01 is the original test's month and the extract that
month really was cut on the 1st, so it exercises only the `created_on` to `visible_on` half
of the fix. 2024-02 is the largest drift in `ch_bulk.MANIFEST`, cut on the 7th, so it is the
month that isolates the `source_date` half. The two halves work against each other: at
2024-02 the visibility half removes 1,243 charges the old clock admitted and the extract-date
half restores 207 of them, for a net 1,036, against 1,455 and zero at 2025-01. About a sixth
of the correction at a drift month is therefore not processing lag at all, it is the date the
file was cut, which is a number `panel.py` has been writing to disk from the start. That
contrast is the case for **decomposing** the 21 days into a known drift plus an estimated
residual rather than tuning a single constant to cover both, which is what stream D is doing.

The same two months also show what the 21-day panel costs: it passes both, but at 2024-02 it
passes by refusing 207 charges that were in that month's extract. Safe, and six days stale.

### Nothing in notebook 16 is compromised by any of this

Worth being explicit, because "I added features to the matrix" is exactly the sort of sentence
that silently invalidates a saved result. Four things were true before this notebook and are
still true:

1. **`data/processed/model_matrix/` was not written to.** Everything here goes to
   `model_matrix_lender/`. I re-derived the `lending` matrix from the original code path and it
   is row-for-row identical to what is on disk, 457,941 x 45.
2. **The recorded baseline was not written to.** Its metrics and its four SHAP importance
   tables are exactly as notebook 16 left them. They have since moved from the flat
   `reports/step6/` into `reports/runs/baseline/`, which is a relocation of the same bytes
   plus a manifest saying which matrix produced them, not a re-run.
3. **`targets.FEATURE_COLS` is still the same 41 columns and `targets.TARGETS` is still the
   same four targets.** The wider list is a separate name (`FEATURE_COLS_LENDER`) and the
   fifth target is in a separate dict (`LENDER_TARGETS`), so notebook 16's loops cannot pick
   either of them up without someone deliberately editing it.
4. **`train.split_origins` returns the same splits for all four targets**, so a re-run of
   notebook 16 today reproduces the same train/test boundaries it used before.

The one line of existing code I did change is in `train.py`: the horizon lookup now reads
`targets.ALL_TARGETS[target]` rather than `targets.TARGETS[target]`. `ALL_TARGETS` is the four
plus `switching`, so every existing lookup resolves to the identical value and a future
switching run can find its horizon without the four-target loops changing behaviour.

What I would do next, in order:

1. **Re-run step 6 against `model_matrix_lender/`** and compare out-of-time AUC to the run
   recorded as the `baseline` run in `reports/runs/` (that is the only run saved today; the
   post-refactor run in notebook 16 has not been written out yet). In notebook 16 this is one
   line, `CFG = train.LENDER`, and `train.compare_runs(["baseline", "lender"])` afterwards.
   That comparison is the only thing that can say whether a day of harvesting bought
   predictive power, as opposed to the client-facing filters above, which it already bought
   outright.
2. **Watch `is_lbg_client` in the SHAP plots.** If it ranks highly on `lending` that is
   interesting and slightly awkward: it would mean Lloyds clients are systematically more
   likely to take new charges, which is as much a statement about who Lloyds already banks as
   about the company.
3. **Leave `switching` untrained**, on the reasoning and the numbers in section 7. Revisit it
   when the panel has another year of history, not by weakening the definition.